In [1]:
# Imports

import sys
import os
import mlflow
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import chromadb
from chromadb.config import Settings

/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as 

In [2]:
p3_root_path = os.path.abspath("..") 

if p3_root_path not in sys.path:
    sys.path.append(p3_root_path)

print(f"Added to path: {p3_root_path}")

try:
    from ingestion import initialize_ingestion
    from pipeline import generate_answer
    print(" P3 Modules loaded successfully.")
except ImportError as e:
    print(f" Still failing: {e}")
    # If it still fails, let's see what's actually there
    print(f"Directory contents: {os.listdir(p3_root_path)}")

Added to path: /home/spectra/Desktop/ml-ai-journey/project3-rag
 P3 Modules loaded successfully.


In [3]:
DATA_FILE = "../apple_10k_2023.htm" # Ensure this path points to your file
COMPANY = "Apple Inc."

print(f"📡 Initializing RAG Engine for {COMPANY}...")

# Call your production ingestion logic
# Note: It already has the guard to skip embedding if chroma_db/ exists
artifacts = initialize_ingestion(file_path=DATA_FILE, company_name=COMPANY)

print("\n--- Artifacts Integrity Check ---")
print(f"✅ Collection: {artifacts['collection'].name} (Count: {artifacts['collection'].count()})")
print(f"✅ BM25 Index: Loaded ({len(artifacts['enriched_texts'])} documents)")
print(f"✅ Model: {artifacts['model'].__class__.__name__} is warm.")

📡 Initializing RAG Engine for Apple Inc....


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2079.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Found existing collection with 74 chunks. Skipping embedding.

--- Artifacts Integrity Check ---
✅ Collection: financial_docs_v2 (Count: 74)
✅ BM25 Index: Loaded (74 documents)
✅ Model: SentenceTransformer is warm.


# Building the Evaluation Dataset Programmatically

In [4]:
import time
from datasets import Dataset

In [5]:
test_queries = [
    {"question": "What was Apple’s total net sales for the 2023 fiscal year?", "ground_truth": "Apple’s total net sales for the 2023 fiscal year were $383,285 million."},
    {"question": "What is the par value per share of Apple’s common stock as reported in the 10-K?", "ground_truth": "The par value per share of common stock is $0.00001."},
    {"question": "In which city and state are Apple’s principal executive offices located?", "ground_truth": "Apple’s principal executive offices are located in Cupertino, California."},
    {"question": "Compare the 'Research and Development' (R&D) expenses between 2022 and 2023. What was the absolute dollar increase, and what primary factors did management cite for this change?", "ground_truth": "R&D expenses increased from $26,251 million in 2022 to $29,915 million in 2023, an absolute increase of $3,664 million. The increase was primarily driven by higher headcount-related expenses and engineering program costs."},
    {"question": "Based on the 'Note 3 – Financial Instruments', how does Apple characterize its concentration of credit risk regarding its cash, cash equivalents, and marketable securities, and what is the typical credit rating of these investments?", "ground_truth": "Apple limits credit risk by restricting exposure to any single issuer and requires investments to be high credit quality, generally rated single-A or higher by S&P and A2 or higher by Moody’s."}
]

In [6]:
eval_samples = []

In [7]:
print("🚀 Starting Production Audit: Generating Answers & Contexts...")

for item in test_queries:
    query = item["question"]
    print(f"  → Querying: {query[:50]}...")
    
    # 1. Capture retrieval state (Contexts)
    # Based on your generation.py snippet, we call the same retrieve logic
    from retrieval import retrieve_with_citations # Using your exact func name
    
    retrieved_docs = retrieve_with_citations(
        query=query,
        bm25_index=artifacts["bm25"],
        documents=artifacts["enriched_texts"],
        chunks=artifacts["chunks"],
        collection=artifacts["collection"]
    )
    # RAGAs needs a list of strings
    contexts = [doc["text"] for doc in retrieved_docs]
    
    # 2. Capture generation state (Answer)
    response = generate_answer(query, artifacts)
    answer = response["answer"]
    
    # 3. Package
    eval_samples.append({
        "question": query,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": item["ground_truth"]
    })
    
    # Avoid hitting Groq Rate Limits (Optional but safer on Arch)
    time.sleep(1)

# Convert to RAGAs-compatible format
ragas_dataset = Dataset.from_list(eval_samples)
print(f"\n✅ Audit complete. Prepared {len(ragas_dataset)} test cases.")

🚀 Starting Production Audit: Generating Answers & Contexts...
  → Querying: What was Apple’s total net sales for the 2023 fisc...
  → Querying: What is the par value per share of Apple’s common ...
  → Querying: In which city and state are Apple’s principal exec...
  → Querying: Compare the 'Research and Development' (R&D) expen...
  → Querying: Based on the 'Note 3 – Financial Instruments', how...

✅ Audit complete. Prepared 5 test cases.


# Ragas Evaluation  execution 

In [ ]:
# Cell 5: RAGAs Evaluation Execution (The Total Independence Version)
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.run_config import RunConfig

# 1. The Judge (LLM)
judge_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key="####",
    temperature=0 # Keep the judge objective
)



# 2. The Critic (Embeddings) 
# We use the same model as your ingestion to keep things consistent
judge_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


run_config = RunConfig(max_workers=1, timeout=60)

print("⚖️ Scoring pipeline... (Local Embeddings + Groq Judge)")

# 3. Execute with explicit LLM and Embeddings
result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        # answer_relevancy, # Temporarily comment this out if the n=1 error persists
        context_precision,
        context_recall,
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
    run_config=run_config
)


df = result.to_pandas()
display_cols = ['user_input', 'response', 'faithfulness', 'context_precision', 'context_recall']
actual_cols = [c for c in display_cols if c in df.columns]

print("\n--- FINAL DAY 68 VERDICT ---")
print(df[actual_cols])

print("\n--- METRIC AVERAGES ---")
print(df[['faithfulness', 'context_precision', 'context_recall']].mean())

import mlflow
with mlflow.start_run(run_name="RAGAs_Audit_Final"):
    # Log average scores
    for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
        if metric in df.columns:
            mlflow.log_metric(f"avg_{metric}", df[metric].mean())
    print("\n✅ Results synced to MLflow.")

/tmp/ipykernel_262060/502335004.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_262060/502335004.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_262060/502335004.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfu

⚖️ Scoring pipeline... (Local Embeddings + Groq Judge)


Evaluating: 100%|██████████| 15/15 [02:04<00:00,  8.33s/it]



--- FINAL DAY 68 VERDICT ---
                                          user_input  \
0  What was Apple’s total net sales for the 2023 ...   
1  What is the par value per share of Apple’s com...   
2  In which city and state are Apple’s principal ...   
3  Compare the 'Research and Development' (R&D) e...   
4  Based on the 'Note 3 – Financial Instruments',...   

                                            response  faithfulness  \
0  The provided sources do not contain the total ...      0.333333   
1  The provided sources do not contain informatio...      1.000000   
2  The provided sources do not contain informatio...      1.000000   
3  In 2023, Research and Development (R&D) expens...      1.000000   
4  The provided sources do not contain informatio...      0.750000   

   context_precision  context_recall  
0           1.000000             1.0  
1           0.000000             0.0  
2           0.000000             0.0  
3           0.333333             0.5  
4           0.000

In [10]:
!mlflow ui

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/14 15:43:53 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/05/14 15:43:53 INFO:     Started parent process [270769]
2026/05/14 15:43:58 INFO:     Started server process [270773]
2026/05/14 15:43:58 INFO:     Waiting for application startup.
2026/05/14 15:43:58 INFO:     Application startup complete.
2026/05/14 15:43:58 INFO:     Started server process [270776]
2026/05/14 15:43:58 INFO:     Waiting for application startup.
2026/05/14 15:43:58 INFO:     Application startup complete.
2026/05/14 15:43:58 INFO:     Started server process [270774]
2026/05/14 15:43:58 INFO:     Waiting for application startup.
2026/05/14 15:43:58 INFO:     Applicatio